# Q6 — Architecture Capacity, Cost and Deployment

Builds a student-designed CNN family and measures parameters, MACs, receptive field, model size, inference latency and validation accuracy. CIFAR-10 is used as a public image-classification proxy; it is not a wildlife dataset.

## 1. Imports, reproducibility and device

In [ ]:
import os,time,random,json,zipfile,gc
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
import torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import DataLoader,Subset
from torchvision import datasets,transforms
from sklearn.metrics import accuracy_score

SEED=42; EPOCHS=2; BATCH=128; LR=0.001
if not torch.cuda.is_available(): torch.set_num_threads(min(4,os.cpu_count() or 1))
def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
seed_all()
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT=Path("Q6_outputs"); OUT.mkdir(exist_ok=True)
print("Device:",device)


## 2. Dataset

The assignment describes wildlife camera-trap classification. This notebook uses CIFAR-10 as a publicly downloadable 10-class image benchmark to validate architecture cost trends. It is a proxy benchmark, not wildlife-camera data; state this clearly. The architecture and cost analysis remain applicable, but wildlife-specific accuracy cannot be claimed.

In [ ]:
# Use a common 32x32 RGB source dataset; resize to each architecture's input resolution.
base_tf=transforms.ToTensor()
train_full=datasets.CIFAR10("./data",train=True,download=True,transform=base_tf)
test_full=datasets.CIFAR10("./data",train=False,download=True,transform=base_tf)
g=torch.Generator().manual_seed(SEED)
perm=torch.randperm(len(train_full),generator=g)
TRAIN_N,VAL_N,TEST_N=3000,600,1000
train_ids=perm[:TRAIN_N].tolist(); val_ids=perm[TRAIN_N:TRAIN_N+VAL_N].tolist()
test_ids=list(range(TEST_N))
classes=train_full.classes
print("Train/val/test:",TRAIN_N,VAL_N,TEST_N)


## 3. Student-designed architecture family

Three custom CNN configurations vary depth, width and input resolution. They are not copied from VGG or ResNet. The model uses repeated 3×3 convolutions, ReLU and 2×2 max pooling, followed by global average pooling and a linear classifier.

In [ ]:
# Architecture specifications: (input resolution, channel widths)
ARCHS={
    "Compact_D2_W16_R32": {"resolution":32,"channels":[16,32]},
    "Balanced_D3_W24_R32": {"resolution":32,"channels":[24,48,64]},
    "Larger_D4_W32_R48": {"resolution":48,"channels":[32,64,96,128]}
}

class CustomCameraCNN(nn.Module):
    def __init__(self,channels,num_classes=10):
        super().__init__()
        layers=[]; in_ch=3
        for out_ch in channels:
            layers += [nn.Conv2d(in_ch,out_ch,kernel_size=3,stride=1,padding=1,bias=True),
                       nn.ReLU(inplace=True),nn.MaxPool2d(2)]
            in_ch=out_ch
        self.features=nn.Sequential(*layers)
        self.gap=nn.AdaptiveAvgPool2d((1,1))
        self.classifier=nn.Linear(in_ch,num_classes)
    def forward(self,x):
        return self.classifier(torch.flatten(self.gap(self.features(x)),1))

def make_loaders(resolution):
    tf=transforms.Compose([transforms.Resize((resolution,resolution)),transforms.ToTensor()])
    # Apply the same deterministic indices to the same base dataset, with resolution-specific transform.
    trbase=datasets.CIFAR10("./data",train=True,download=False,transform=tf)
    tebase=datasets.CIFAR10("./data",train=False,download=False,transform=tf)
    tr=Subset(trbase,train_ids); va=Subset(trbase,val_ids); te=Subset(tebase,test_ids)
    gen=torch.Generator().manual_seed(SEED)
    return (DataLoader(tr,batch_size=BATCH,shuffle=True,num_workers=0,generator=gen),
            DataLoader(va,batch_size=BATCH,shuffle=False,num_workers=0),
            DataLoader(te,batch_size=BATCH,shuffle=False,num_workers=0))


## 4. Cost equations and hand-verification utilities

For a convolutional layer:

\[
P=(K_hK_wC_{in}+1)C_{out}
\]

\[
MACs=H_{out}W_{out}C_{out}(K_hK_wC_{in})
\]

The receptive field and effective stride can be updated layer by layer:

\[
r_l=r_{l-1}+(k_l-1)j_{l-1},\quad j_l=j_{l-1}s_l
\]

where \(r\) is receptive field and \(j\) is the input-pixel jump between adjacent features. This notebook counts one multiply-accumulate as one MAC.

In [ ]:
def conv_output(h,k=3,s=1,p=1): return (h+2*p-k)//s+1

def manual_cost(channels,resolution):
    h=w=resolution; in_ch=3; total_params=0; total_macs=0
    rf=1; jump=1; rows=[]
    for i,out_ch in enumerate(channels,1):
        # Conv 3x3, stride 1, padding 1
        conv_h=conv_output(h); conv_w=conv_output(w)
        params=(3*3*in_ch+1)*out_ch
        macs=conv_h*conv_w*out_ch*(3*3*in_ch)
        rf=rf+(3-1)*jump
        # MaxPool 2x2 stride 2
        rf=rf+(2-1)*jump
        jump=jump*2
        h,w=conv_h//2,conv_w//2
        rows.append({"Layer":i,"Output_after_pool":f"{h}x{w}x{out_ch}",
                     "Parameters":params,"Conv_MACs":macs,"Receptive_Field":rf,"Jump":jump})
        total_params+=params; total_macs+=macs; in_ch=out_ch
    # global average pool and dense classifier
    dense_params=in_ch*10+10
    total_params+=dense_params
    rows.append({"Layer":"Classifier","Output_after_pool":"10","Parameters":dense_params,
                 "Conv_MACs":in_ch*10,"Receptive_Field":rf,"Jump":jump})
    total_macs+=in_ch*10
    return rows,total_params,total_macs

for name,spec in ARCHS.items():
    rows,p,m=manual_cost(spec["channels"],spec["resolution"])
    print("\\n",name,"manual params:",p,"manual MACs:",m)
    display(pd.DataFrame(rows))


## 5. Verify framework parameter counts, MACs, and output shapes

The hand calculation is compared against PyTorch parameter counts and forward-hook MAC counts. The hook counts convolution and linear operations; pooling and activation comparisons are not counted as MACs.

In [ ]:
def framework_macs(model,input_shape):
    macs={"total":0}; shapes={}
    def hook(module,inputs,output):
        x=inputs[0]
        if isinstance(module,nn.Conv2d):
            kh,kw=module.kernel_size
            macs["total"]+=output.shape[0]*output.shape[2]*output.shape[3]*module.out_channels*(kh*kw*module.in_channels//module.groups)
            shapes[module]=tuple(output.shape)
        elif isinstance(module,nn.Linear):
            macs["total"]+=output.numel()*module.in_features
            shapes[module]=tuple(output.shape)
    hooks=[]
    for m in model.modules():
        if isinstance(m,(nn.Conv2d,nn.Linear)): hooks.append(m.register_forward_hook(hook))
    with torch.no_grad(): out=model(torch.zeros(*input_shape).to(device))
    for h in hooks: h.remove()
    return macs["total"],tuple(out.shape)

verification=[]
for name,spec in ARCHS.items():
    model=CustomCameraCNN(spec["channels"]).to(device)
    manual_rows,manual_params,manual_macs=manual_cost(spec["channels"],spec["resolution"])
    framework_params=sum(p.numel() for p in model.parameters())
    framework_mac_count,outshape=framework_macs(model,(1,3,spec["resolution"],spec["resolution"]))
    verification.append({"Architecture":name,"Manual_Params":manual_params,
      "Framework_Params":framework_params,"Manual_MACs":manual_macs,
      "Framework_MACs":framework_mac_count,"Output_Shape":str(outshape),
      "Parameter_Check":manual_params==framework_params,"MAC_Check":manual_macs==framework_mac_count})
display(pd.DataFrame(verification))


## 6. Train, validate and measure deployment quantities

A short, equal training budget is used for all three configurations. Accuracy is a validation-set measurement. Model file size is measured from the saved state dictionary. Inference time is measured per image after warm-up; timing depends on hardware and runtime load.

In [ ]:
criterion=nn.CrossEntropyLoss()
def evaluate(model,loader):
    model.eval(); preds=[]; actual=[]; loss_sum=0
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(device),y.to(device); logits=model(x)
            loss_sum+=criterion(logits,y).item()*len(y)
            preds.extend(logits.argmax(1).cpu().tolist()); actual.extend(y.cpu().tolist())
    return loss_sum/len(loader.dataset),accuracy_score(actual,preds)

def train_one(model,tr,va):
    opt=torch.optim.Adam(model.parameters(),lr=LR); history=[]
    if device.type=="cuda": torch.cuda.synchronize()
    start=time.perf_counter()
    for ep in range(1,EPOCHS+1):
        model.train(); total=0
        for x,y in tr:
            x,y=x.to(device),y.to(device); opt.zero_grad(set_to_none=True)
            loss=criterion(model(x),y); loss.backward(); opt.step(); total+=loss.item()*len(y)
        vl,va_acc=evaluate(model,va)
        history.append({"epoch":ep,"train_loss":total/len(tr.dataset),"val_loss":vl,"val_accuracy":va_acc})
        print(f"Epoch {ep}/{EPOCHS}: val loss={vl:.4f}, val accuracy={va_acc:.4f}")
    if device.type=="cuda": torch.cuda.synchronize()
    return history,time.perf_counter()-start

def measure_latency(model,resolution,repeats=100):
    model.eval(); x=torch.randn(1,3,resolution,resolution).to(device)
    with torch.no_grad():
        for _ in range(10): model(x)
        if device.type=="cuda": torch.cuda.synchronize()
        start=time.perf_counter()
        for _ in range(repeats): model(x)
        if device.type=="cuda": torch.cuda.synchronize()
    return (time.perf_counter()-start)/repeats*1000

results=[]; histories={}
for name,spec in ARCHS.items():
    print("\\nTraining",name); seed_all()
    tr,va,te=make_loaders(spec["resolution"])
    model=CustomCameraCNN(spec["channels"]).to(device)
    hist,train_time=train_one(model,tr,va)
    val_loss,val_acc=evaluate(model,va)
    macs,outshape=framework_macs(model,(1,3,spec["resolution"],spec["resolution"]))
    weight_path=OUT/f"{name}.pth"; torch.save(model.state_dict(),weight_path)
    size_mb=weight_path.stat().st_size/(1024**2)
    latency=measure_latency(model,spec["resolution"])
    params=sum(p.numel() for p in model.parameters())
    results.append({"Architecture":name,"Depth":len(spec["channels"]),"Width_max":max(spec["channels"]),
      "Resolution":spec["resolution"],"Parameters":params,"MACs":macs,
      "Model_Size_MB":size_mb,"Inference_ms_per_image":latency,
      "Training_Time_s":train_time,"Validation_Loss":val_loss,"Validation_Accuracy":val_acc})
    histories[name]=hist
    pd.DataFrame(results).to_csv(OUT/"Q6_checkpoint.csv",index=False)
    torch.save(model.state_dict(),OUT/f"{name}_weights.pth")
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

results_df=pd.DataFrame(results); display(results_df)
results_df.to_csv(OUT/"Q6_results.csv",index=False)


## 7. Growth investigation: doubling depth, width and resolution

For a convolution, MACs scale approximately with \(H W C_{in} C_{out} K^2\). Doubling both input and output channels in a layer gives roughly 4× convolutional parameters and MACs. Doubling both spatial dimensions gives roughly 4× MACs while leaving parameter count unchanged for that layer. Doubling the number of similar layers gives roughly 2× cost, if layer widths and spatial sizes remain comparable. Actual whole-network scaling depends on where downsampling occurs and how channels are connected.

In [ ]:
# Analytical illustration for one representative convolution: Cin=Cout=32, K=3, bias included.
def one_conv_cost(H,W,Cin,Cout,K=3):
    params=(K*K*Cin+1)*Cout
    macs=H*W*Cout*K*K*Cin
    return params,macs

base=one_conv_cost(32,32,32,32)
scenarios={
 "Base":base,
 "Double depth (2 similar layers)":(2*base[0],2*base[1]),
 "Double width (Cin and Cout)":one_conv_cost(32,32,64,64),
 "Double resolution (H and W)":one_conv_cost(64,64,32,32)
}
growth=[]
for k,(p,m) in scenarios.items():
    growth.append({"Scenario":k,"Parameters":p,"MACs":m,
                   "Parameter_Factor_vs_Base":p/base[0],"MAC_Factor_vs_Base":m/base[1]})
growth_df=pd.DataFrame(growth); display(growth_df)


## 8. Deployment boundary and export

For a reproducible decision rule, this notebook uses illustrative limits of 5 MB model storage and 2 ms per-image inference. These are experimental assumptions, not measured properties of a particular wildlife camera. Change them to match the target hardware if known. Select the highest-validation-accuracy architecture satisfying both limits.

In [ ]:
# Edit these budgets to reflect the actual device if its specification is known.
MEMORY_BUDGET_MB=5.0
LATENCY_BUDGET_MS=2.0

eligible=results_df[(results_df["Model_Size_MB"]<=MEMORY_BUDGET_MB) &
                    (results_df["Inference_ms_per_image"]<=LATENCY_BUDGET_MS)]
if len(eligible):
    chosen=eligible.sort_values("Validation_Accuracy",ascending=False).iloc[0]
    largest=results_df.sort_values("Parameters",ascending=False).iloc[0]
    print("Recommended under assumed budget:",chosen["Architecture"])
    print("Validation accuracy sacrificed vs largest model:",
          float(largest["Validation_Accuracy"]-chosen["Validation_Accuracy"]))
else:
    print("No architecture meets both assumed limits. Revise the architecture or device budget.")
    chosen=None

plt.figure(figsize=(8,5))
plt.scatter(results_df["MACs"],results_df["Validation_Accuracy"],s=80)
for _,r in results_df.iterrows(): plt.annotate(r["Architecture"],(r["MACs"],r["Validation_Accuracy"]))
plt.xlabel("MACs per image"); plt.ylabel("Validation accuracy")
plt.title("Accuracy versus computational cost"); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(OUT/"Q6_accuracy_vs_macs.png",dpi=150); plt.show()

with open(OUT/"Q6_summary.json","w") as f:
    json.dump({"device":str(device),"seed":SEED,"epochs":EPOCHS,"batch_size":BATCH,
               "memory_budget_MB":MEMORY_BUDGET_MB,"latency_budget_ms":LATENCY_BUDGET_MS,
               "results":results,"verification":verification,"growth":growth},f,indent=2)

with zipfile.ZipFile("Q6_complete_evidence.zip","w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,arcname=p.relative_to(OUT.parent))
print("Created Q6_complete_evidence.zip")
try:
    from google.colab import files
    files.download("Q6_complete_evidence.zip")
except ImportError: pass
